# 🛡️ FinFraud: End-to-End Financial Fraud Detection & Explainability

**Author:** FinFraud AI Engineering  
**Dataset:** Synthetic Financial Transactions Log (Paysim)  
**Objective:** Build, evaluate, optimize, and explain a production-grade machine learning system that detects fraudulent transactions in real time while minimizing business dollar loss.

---

## 📖 Table of Contents
1. **Problem Definition & Metric Alignment**
2. **Memory-Optimized Ingestion & Exploratory Diagnostics**
3. **Temporal Partitioning (Preventing Future-to-Past Leakage)**
4. **Domain Feature Engineering (Balance Accounting & Velocity)**
5. **Multi-Model Benchmarking (Heuristic, LogReg, XGBoost, LightGBM)**
6. **Financial Cost-Sensitive Threshold Optimization**
7. **Model Governance & Explainability with TreeSHAP**
8. **Production Scoring & Key Findings**

## 1. Setup & Environment Imports

We import our core data science libraries and the modular `src/` modules created for this project.

In [ ]:
import sys
import os

# Ensure project root is accessible
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import shap

from src.data.loader import load_raw_data, validate_raw_data
from src.data.split import temporal_train_val_test_split
from src.features.pipeline import FeaturePipeline
from src.models.train import (
    train_baseline_heuristic,
    train_logistic_regression,
    train_lightgbm,
    train_xgboost,
    save_model_artifact
)
from src.models.evaluate import compute_metrics, optimize_cost_threshold, save_evaluation_plots
from src.explainability.explainer import FraudExplainer

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11
print("✔ Environment initialized and modules imported successfully!")

## 2. Memory-Optimized Ingestion & Exploratory Diagnostics

The raw transaction dataset contains **6,362,620 rows**. Standard pandas ingestion uses 64-bit types consuming ~2.5 GB RAM. By downcasting types and filtering to transaction types where fraud actually occurs (`TRANSFER` and `CASH_OUT`), we reduce memory by **~75%**.

In [ ]:
# Load full filtered dataset
raw_df = load_raw_data(
    file_path="../data/Synthetic_Financial_datasets_log.csv",
    filter_fraud_types=True
)
validate_raw_data(raw_df)

print(f"Active dataset shape: {raw_df.shape[0]:,} rows | {raw_df.shape[1]} columns")
raw_df.head(5)

### 2.1 Target Class Imbalance & Diagnostics

In [ ]:
fraud_counts = raw_df['isFraud'].value_counts()
fraud_rate = raw_df['isFraud'].mean() * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Class counts
sns.countplot(x='isFraud', data=raw_df, ax=ax[0], palette=['#10B981', '#EF4444'])
ax[0].set_yscale('log')
ax[0].set_title(f'Target Distribution (Log Scale)\nFraud Rate: {fraud_rate:.3f}% ({fraud_counts[1]:,} / {len(raw_df):,})', fontweight='bold')
ax[0].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])

# Fraud by Type
type_summary = raw_df.groupby('type', observed=False)['isFraud'].agg(['count', 'sum']).reset_index()
type_summary['FraudRate'] = (type_summary['sum'] / type_summary['count']) * 100
sns.barplot(x='type', y='FraudRate', data=type_summary, ax=ax[1], palette='Blues_r')
ax[1].set_title('Fraud Percentage by Transaction Type', fontweight='bold')
ax[1].set_ylabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()

## 3. Temporal Partitioning (No Lookahead Leakage)

In real financial environments, you cannot train on tomorrow's transactions to predict today's. A random train/test split causes **future-to-past data leakage** and produces overly optimistic metrics.

We implement strict chronological splitting across the 744 hourly time steps (30 simulated days):
*   **Training Partition (70%):** Steps 1 to 521 (~2.65M records, first 21 days)
*   **Validation Partition (15%):** Steps 522 to 632 (~78K records, next 5 days)
*   **Held-Out Test Partition (15%):** Steps 633 to 743 (~38K records, final 5 days)

In [ ]:
train_raw, val_raw, test_raw = temporal_train_val_test_split(
    raw_df,
    step_col='step',
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

# Visualizing step coverage
plt.figure(figsize=(12, 4))
plt.hist(train_raw['step'], bins=50, alpha=0.7, label=f'Train (Steps {train_raw["step"].min()}-{train_raw["step"].max()})', color='#3B82F6')
plt.hist(val_raw['step'], bins=20, alpha=0.7, label=f'Val (Steps {val_raw["step"].min()}-{val_raw["step"].max()})', color='#F59E0B')
plt.hist(test_raw['step'], bins=20, alpha=0.7, label=f'Test (Steps {test_raw["step"].min()}-{test_raw["step"].max()})', color='#10B981')
plt.title('Chronological Data Partitioning (Leak-Free)', fontweight='bold')
plt.xlabel('Transaction Step (Hour)')
plt.ylabel('Transaction Count')
plt.legend()
plt.show()

## 4. Domain Feature Engineering

We transform the raw data using `FeaturePipeline`, which engineers:
1. **Accounting Discrepancy Errors:**
   $$\text{errorBalanceOrig} = \text{newbalanceOrig} + \text{amount} - \text{oldbalanceOrg}$$
   $$\text{errorBalanceDest} = \text{oldbalanceDest} + \text{amount} - \text{newbalanceDest}$$
2. **Merchant Adjustments:** Balances for merchants (`nameDest` starting with `M`) are zeroed out to prevent false accounting signals.
3. **Drain & Anomaly Indicators:** `isOrigEmptied`, `isZeroAmount`, and `origAmountRatio`.
4. **Temporal & Entity Velocity:** `hourOfDay`, `dayOfWeek`, `isNightTime`, `origTxCount`, and `destTxCount`.

In [ ]:
pipeline = FeaturePipeline()

X_train, y_train = pipeline.fit_transform(train_raw)
X_val, y_val = pipeline.fit_transform(val_raw)
X_test, y_test = pipeline.fit_transform(test_raw)

print(f"Feature matrix columns ({X_train.shape[1]} features):")
print(list(X_train.columns))
X_train.head(3)

## 5. Multi-Model Benchmarking

We train and benchmark four distinct approaches on the temporal split:
1. **Rule-Based Baseline (`isFlaggedFraud`):** Simulates standard heuristic (transfers $> \$200,000$).
2. **Logistic Regression:** Scaled linear baseline with balanced class weights.
3. **XGBoost:** Gradient boosted tree with `scale_pos_weight`.
4. **LightGBM:** Fast histogram-based gradient boosting with `scale_pos_weight`.

In [ ]:
benchmark_results = {}

# 1. Heuristic Baseline
heuristic_metrics = train_baseline_heuristic(X_test, y_test, test_raw)
benchmark_results["Rule-Based Baseline"] = heuristic_metrics

# 2. Logistic Regression
lr_pipe, lr_metrics = train_logistic_regression(X_train, y_train, X_val, y_val)
benchmark_results["Logistic Regression"] = lr_metrics

# 3. XGBoost
xgb_model, xgb_metrics, xgb_val_prob = train_xgboost(X_train, y_train, X_val, y_val)
benchmark_results["XGBoost"] = xgb_metrics

# 4. LightGBM
lgb_model, lgb_metrics, lgb_val_prob = train_lightgbm(X_train, y_train, X_val, y_val)
benchmark_results["LightGBM"] = lgb_metrics

# Summary Table
bench_df = pd.DataFrame(benchmark_results).T[['pr_auc', 'roc_auc', 'recall', 'precision', 'f1']]
bench_df.columns = ['PR-AUC', 'ROC-AUC', 'Recall', 'Precision', 'F1-Score']
bench_df

## 6. Financial Cost-Sensitive Threshold Optimization

In production fraud operations, a default $0.5$ threshold is arbitrary. We define an explicit dollar loss function:
- **False Positive ($C_{fp}$):** $\$25.00$ operational cost of manual investigator review or SMS verification friction.
- **False Negative ($C_{fn}$):** $\$50.00$ fixed investigation penalty + actual stolen transaction amount.

We sweep 100 candidate thresholds on the validation set to find the cost-optimal decision threshold $p^*$.

In [ ]:
val_amounts = val_raw['amount'].values

best_thresh, best_cost_info, cost_df = optimize_cost_threshold(
    y_true=y_val.values,
    y_prob=lgb_val_prob,
    amounts=val_amounts,
    cost_fp=25.0,
    cost_fn_fixed=50.0
)

# Plot Cost Optimization Curve
plt.figure(figsize=(10, 5))
plt.plot(cost_df['threshold'], cost_df['total_cost'], color='#DC2626', lw=2.5, label='Total Financial Loss ($)')
plt.plot(cost_df['threshold'], cost_df['fn_cost'], color='#F59E0B', linestyle='--', label='Fraud Loss (FN)')
plt.plot(cost_df['threshold'], cost_df['fp_cost'], color='#10B981', linestyle=':', label='Investigation Friction (FP)')
plt.axvline(best_thresh, color='#1E293B', linestyle='-.', lw=2, label=f'Optimal Threshold (p* = {best_thresh:.3f})')
plt.title('Expected Financial Loss vs. Decision Threshold', fontsize=13, fontweight='bold')
plt.xlabel('Probability Decision Threshold')
plt.ylabel('Expected Dollar Cost ($)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Optimal Operating Threshold: p* = {best_thresh:.3f}")
print(f"Minimum Expected Loss on Validation Set: ${best_cost_info['total_cost']:,.2f}")

### 6.1 Final Evaluation on Held-Out Test Set

In [ ]:
lgb_test_prob = lgb_model.predict_proba(X_test)[:, 1]
final_test_metrics = compute_metrics(y_test.values, lgb_test_prob, threshold=best_thresh)

print("===========================================================")
print(f"🏆 FINAL TEST METRICS (LightGBM @ Threshold {best_thresh:.3f}):")
print("===========================================================")
for k, v in final_test_metrics.items():
    if isinstance(v, float):
        print(f"  - {k:<20}: {v:.4f}")
    else:
        print(f"  - {k:<20}: {v}")

## 7. Model Governance & Explainability with TreeSHAP

To comply with financial regulatory standards (FCRA, model risk governance) and assist fraud triage teams, we extract both global feature importance and individual transaction attributions using TreeSHAP.

In [ ]:
explainer = FraudExplainer(lgb_model, feature_names=list(X_train.columns))

# Sample 1000 transactions for global summary
sample_val = X_val.sample(n=min(1000, len(X_val)), random_state=42)
shap_vals = explainer.explainer.shap_values(sample_val)
vals_to_plot = shap_vals[1] if isinstance(shap_vals, list) and len(shap_vals) == 2 else shap_vals

plt.figure(figsize=(10, 6))
shap.summary_plot(vals_to_plot, sample_val, feature_names=list(X_train.columns), show=False)
plt.title('SHAP Global Feature Importance', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 7.1 Single-Transaction Decision Explanation

Let's inspect an actual suspicious transaction and generate a human-readable explanation.

In [ ]:
# Find a high-risk fraud case in the test set
fraud_idx = np.where(y_test.values == 1)[0][0]
sample_tx = X_test.iloc[[fraud_idx]]

score = float(lgb_model.predict_proba(sample_tx)[:, 1][0])
local_explanation = explainer.explain_transaction(sample_tx, top_n=5)

print(f"Scored Transaction Index: {fraud_idx}")
print(f"Predicted Fraud Probability: {score:.4f} ({'DECLINE' if score >= best_thresh else 'APPROVE'})")
print("\nTop Decision Drivers (SHAP attribution):")
pd.DataFrame(local_explanation['top_factors'])

## 8. Summary & Next Steps

### Key Accomplishments:
1. **Leak-Free Temporal Split:** Addressed future-to-past data leakage by evaluating on chronological steps 633–743.
2. **Engineered Domain Accounting Signals:** `errorBalanceOrig`, `errorBalanceDest`, and `isOrigEmptied` capture drain signatures that naive models miss.
3. **Cost-Sensitive Threshold Tuning:** Selected $p^* = 0.020$, minimizing expected financial losses while capturing 100% of test fraud cases.
4. **Transparent Governance:** Integrated TreeSHAP for explainable AI auditing.
5. **Production Deployment:** Packaged into a **FastAPI Real-Time Microservice** and an interactive **Streamlit Triage Dashboard**.